# Datamine HOLES3D Process Example

This notebook demonstrates how to configure and run the **`holes3d`** process wrapper in `dmstudio`.

## Process Description

## HOLES3D

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

**HOLES3D** converts a set of downhole sample data, collars location data and optionally downhole survey data, into a 'desurveyed' static drillholes file in which each sample is identified by its location (X,Y,Z coordinates) and direction (azimuth and dip) in space.

**Note** : HOLES3D is used by **[Drillhole Importer](<../COMMON/DrillholeImporter-screen.md>)**.

In the 'desurveyed' form, each drillhole sample is identified independently in 3D space without reference to its neighbours.

If two or more downhole sample files are specified (maximum 10) then they are merged so that all divisions of all sets of samples are maintained. A typical use of merging is to add an assay file to a lithology file, where the lithology intervals do not match the assay sample boundaries. Another example is to add absent data samples into holes by making the second file a single record per hole defining the total hole length.

The output from the **HOLES3D** process is in the standard drillhole format which is used elsewhere in your application. For example the drillholes can be viewed interactively, composited downhole, and used to interpolate grades into a block model.

### Using Downhole Survey Data

The downhole survey data is optional. If a survey file is not specified it is assumed that all the drillholes are vertical. If the survey file only includes a subset of the total number of drillholes, then it is assumed that all drillholes which are not included in the survey file are vertical. In both cases a warning message is displayed. The positions at which the survey readings are recorded does not have to correspond to sample positions.

**Important** : A maximum of 50,000 survey measurements can be processed for each hole. If the number of points exceeds 50,000 then a subset will be selected so that the total number of points is less than 50,000.

The IDs of all holes that use a subset of survey points are displayed with the progress messages in the Command Window. The IDs are also added to the **ERRORS** file. The **HOLESMRY** file shows the number of survey points processed for all holes.

Each drillhole in the downhole survey data file should have a survey measurement at its collar (AT=0). If this is not the case then the process will automatically move the first survey measurement to the collar position. A warning is issued and a list of all offending holes is displayed. If moving the survey position is not appropriate then you should edit the survey data file and rerun the process.

The process works by first computing a set of positions in space at known coordinates down each hole, and then interpolating between these known points for the top and bottom of each sample. The interpolation uses arcs of circles separately in horizontal and vertical planes.

The process will optionally calculate the XYZ coordinates of the start and end of each sample. This is controlled by parameter **ENDPOINT** , which if set to 1 will create the six extra fields. These coordinates can be useful for creating a DTM of the top or bottom of a seam or stratum.

As well as creating the desurveyed file, the process will optionally create two other files which assist in validating the input data. The **HOLESMRY** file contains a summary of the drillholes in each of the input files. It shows the number of records in each input file for each drillhole. The **ERRORS** file contains a list of:

* surveys from the Downhole Survey file,

* samples from the Downhole Sample file(s),

* collars from the Collars file

which do not pass the validation tests. The tests are detailed in the description of the **ERRORS** file.

If the **HOLESMRY** file shows that one or more of the input data files do not contain entries for every drillhole then a warning is displayed. A warning is also issued if there are any entries in the **ERRORS** file. In order to correct any errors it will be necessary for you to edit the input data files and rerun the process.

You are encouraged to use the optional **HOLESMRY** and **ERRORS** files. You should also take careful note of the output display to see whether any warnings have been issued. If there are any warnings it is strongly recommended that you fix the data problems before using the desurveyed file for subsequent processing.

#### Missing Survey Records

* Survey records in the collar table (in the **DIP** and **BRG** columns) are used (when available) if no other survey records are found for that hole. For example, if survey records exist in both the collar table and survey table for a hole, only records in the survey table are used (and collar records for that hole are ignored).

* If no survey records exist for a hole in the survey table, the DIP and BRG values from the collar table are used.

* If the collar table contains absent DIP or BRG records, the hole is set vertically.

### The SURVSMTH Parameter

When a hole sample is desurveyed the survey data (azimuth and dip) of the sample is used to locate the sample centre point in space. A desurveyed drillhole file contains a set of samples each with a calculated center point in XYZ world space.

Sometimes raw drillhole data tables to be desurveyed may contain more than one survey record within one sample, each with different azimuth and dips. Since a sample is by definition a straight line its location in space cannot be calculated using more than one survey record. The SURVSMTH parameter is used to automatically divide up samples where more than one survey records lie within a sample.

The samples are split in half until only one survey record lies within each sample. Therefore many samples may be created. The default value of SURVSMTH is 1 which will cause extra samples to be created so that no sample contains more than one survey record within its **FROM** and **TO** values. For no extra samples to be created the SURVSMTH parameter should be set to zero.

If the SURVSMTH parameter is set to zero and a sample does contain more than one survey record not all survey records will be taken into account. Traditionally this has been resolved by first compositing the samples to reduce their lengths. The SURVSMTH parameter avoids this requirement.

It is often the case that the first one or two samples in exploration holes contain more than one survey record because they are relatively long. This is because sample divisions have not had to have been created through assay and lithological identification near the surface

### Input Files:

* **collar** (Collars):
  Data file of drillhole collar locations. Expects fields **BHID** , **XCOLLAR** ,
  **YCOLLAR** and **ZCOLLAR** ; optional fields **BRG** , **DIP**. If **BRG** and **DIP**
  exist, these values will only be used when no valid survey records exist in input file
  **SURVEY**. If **BRG** and **DIP** values in **COLLAR** are absent or these columns are
  missing, holes are presumed vertical.
  Required=Yes

* **survey** (Downhole Survey):
  Optional survey data file. Expects fields **BHID** , **AT** , **BRG** , **DIP**. If a
  borehole has Survey Data, then it must include a record for the collar location, i.e.
  **AT** =0. **DIP** /**BRG** will first be taken from **SURVEY** file, if that does not
  exist **DIP** /**BRG** will be taken from **COLLAR** and any other holes are presumed
  vertical.
  Required=No

* **sample1** (Downhole Sample):
  Sample data files. This file is compulsory and must include fields **BHID** , **FROM**
  , and **TO**. It will probably also include at least one sample attribute field, such as
  grade or lithology.
  Required=Yes

* **sample2** (Downhole Sample):
  Sample data files. This file is compulsory and must include fields **BHID** , **FROM**
  , and **TO**. It will probably also include at least one sample attribute field, such as
  grade or lithology.
  Required=Yes

* **sample3** (Downhole Sample):
  Sample data files. This file is compulsory and must include fields **BHID** , **FROM**
  , and **TO**. It will probably also include at least one sample attribute field, such as
  grade or lithology.
  Required=Yes

* **sample4** (Downhole Sample):
  Sample data files. This file is compulsory and must include fields **BHID** , **FROM**
  , and **TO**. It will probably also include at least one sample attribute field, such as
  grade or lithology.
  Required=Yes

* **sample5** (Downhole Sample):
  Sample data files. This file is compulsory and must include fields **BHID** , **FROM**
  , and **TO**. It will probably also include at least one sample attribute field, such as
  grade or lithology.
  Required=Yes

* **sample6** (Downhole Sample):
  Sample data files. This file is compulsory and must include fields **BHID** , **FROM**
  , and **TO**. It will probably also include at least one sample attribute field, such as
  grade or lithology.
  Required=Yes

* **sample7** (Downhole Sample):
  Sample data files. This file is compulsory and must include fields **BHID** , **FROM**
  , and **TO**. It will probably also include at least one sample attribute field, such as
  grade or lithology.
  Required=Yes

* **sample8** (Downhole Sample):
  Sample data files. This file is compulsory and must include fields **BHID** , **FROM**
  , and **TO**. It will probably also include at least one sample attribute field, such as
  grade or lithology.
  Required=Yes

* **sample9** (Downhole Sample):
  Sample data files. This file is compulsory and must include fields **BHID** , **FROM**
  , and **TO**. It will probably also include at least one sample attribute field, such as
  grade or lithology.
  Required=Yes

* **sample10** (Downhole Sample):
  Sample data files. This file is compulsory and must include fields **BHID** , **FROM**
  , and **TO**. It will probably also include at least one sample attribute field, such as
  grade or lithology.
  Required=Yes

### Output Files:

* **out** (Drillhole):
  Required=Yes

* **holesmry** (Table):
  Required=No

* **errors**:
  Required=No

### Fields:

* **bhid** (Any : COLLAR, SAMPLE1, SURVEY):
  Drillhole identifier.
  Default=BHID
  Required=No

* **xcollar** (Numeric : COLLAR):
  X co-ordinate of drillhole collar.
  Default=XCOLLAR
  Required=No

* **ycollar** (Numeric : COLLAR):
  Y co-ordinate of drillhole collar.
  Default=YCOLLAR
  Required=No

* **zcollar** (Numeric : COLLAR):
  Z co-ordinate of drillhole collar.
  Default=ZCOLLAR
  Required=No

* **from** (Numeric : SAMPLE1):
  Downhole distance to sample top.
  Default=FROM
  Required=No

* **to** (Numeric : SAMPLE1):
  Downhole distance to sample bottom.
  Default=TO
  Required=No

* **at** (Numeric : SURVEY):
  Downhole distance to survey point.
  Default=AT
  Required=No

* **brg** (Numeric : SURVEY):
  Bearing of drillhole.
  Default=BRG
  Required=No

* **dip** (Numeric : SURVEY):
  Dip of drillhole. Dip values must always be positive when referring to the downwards
  direction if using this command in a batch process. For more information on Downhole
  Survey files, click [here](<../COMMON/filetype.md#Survey>).
  Default=DIP
  Required=No

### Parameters:

* **survsmth**:

* **Options** (0: Prevent samples being added to the output file.):
  Range=
  Values=
  Default=add samples where there are more than one survey record per sample.
  Required=No

* **endpoint**:
  set to 1 to include the **X** , **Y** and **Z** coordinates of the start and end of
  each sample in the desurveyed output file. Fields **XSTART** , **YSTART** , **ZSTART** ,
  **XEND** , **YEND** and **ZEND** are created in the output file.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **desurvmd**:
  Locate sample centers or end points on the desurveyed arcs. The default is 1, to
  accurately locate the sample center points. =0 : To accurately locate sample **END**
  points on the desurveyed arcs. =1 : To accurately locate sample **CENTER** points on the
  desurveyed arcs.
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **dipmeth**:
  Set to 1 to ensure that positive dip values point downwards, or -1 to point upwards.
  See "Defining DIP and BeaRinG", above.
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **inclmiss**:
  INCLude MISSing samples parameter.
  Range=
  Values=
  Default=
  Required=No

* **prompt**:
  Set to 1 (default) to pause **HOLES3D** execution if an error occurs. Set to 0 to
  continue processing script if errors are encountered (useful when running **HOLES3D**
  from script as processing will continue).
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **keepname**:
  Determine how field names are treated during processing.
  Range=
  Values=
  Default=
  Required=No

* **print**:

* **Options** (1: to display each individual process which is run by the **HOLES3D**):
  superprocess.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('holes3d')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute holes3d
print("Running holes3d...")
dm_cmd.holes3d(
    collar_i='t_collars',  # required
    samples_i=['t_assays'],  # required
    out_o='t_holes3d_out',  # required
    # survey_i='t_surveys',  # optional
    # holesmry_o='t_holes3d_summary',  # optional
    # errors_o='t_holes3d_errors',  # optional
    # bhid_f='BHID',  # optional
    # xcollar_f='XCOLLAR',  # optional
    # ycollar_f='YCOLLAR',  # optional
    # zcollar_f='ZCOLLAR',  # optional
    # from_f='FROM',  # optional
    # to_f='TO',  # optional
    # at_f='AT',  # optional
    # brg_f='BRG',  # optional
    # dip_f='DIP',  # optional
    # survsmth_p='add samples where there are more than one survey record per sample.',  # optional
    # endpoint_p=0,  # optional
    # desurvmd_p=1,  # optional
    # dipmeth_p=1,  # optional
    # inclmiss_p='optional',  # optional
    # prompt_p=1,  # optional
    # keepname_p='optional',  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("holes3d execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_holes3d_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")